In [20]:
import os
import pandas as pd

# Define the root folder
root_folder = "smartbugs-curated-v2"  # Change this to your target folder

# Initialize a list to store data
data = []

# Walk through the directory
for subdir, _, files in os.walk(root_folder):
    subfolder_name = os.path.basename(subdir)  # Get the name of the subfolder
    for file in files:
        if ".sol" in file:
            data.append([subfolder_name, file])

# Create DataFrame
df = pd.DataFrame(data, columns=["subfolder", "filename"])

In [22]:
df

,subfolder,filename
0,access_control,arbitrary_location_write_simple.sol
1,access_control,FibonacciBalance.sol
2,access_control,incorrect_constructor_name1.sol
3,access_control,incorrect_constructor_name2.sol
4,access_control,incorrect_constructor_name3.sol
...,...,...
142,unchecked_low_level_calls,etherpot_lotto.sol
143,unchecked_low_level_calls,king_of_the_ether_throne.sol
144,unchecked_low_level_calls,lotto.sol
145,unchecked_low_level_calls,mishandled.sol


In [24]:
import re

cwd = os.getcwd()
parent_folder = f"{cwd}\\smartbugs-curated-v2"

def extract_vulnerable_lines(source_code):
    match = re.search(r'@vulnerable_at_lines:\s*([\d,\s]+)', source_code)
    if match:
        line_numbers = set(map(int, match.group(1).split(',')))
        return line_numbers
    return set()

def read_file(path):
    with open(path, 'r', encoding='utf-8') as file:
        return file.read()

def extract_and_update_vulnerable_lines_in_df(df):
    df['source_code'] = None
    df['vuln_lines'] = None
    for index, row in df.iterrows():
        file_path = f"{parent_folder}\\{row['subfolder']}\\{row['filename']}"
        content = read_file(file_path)
        if content is not None:
            line_numbers = extract_vulnerable_lines(content)
            if len(line_numbers) > 0:
                df.loc[index, "source_code"] = content
                df.loc[index, "vuln_lines"] = str(line_numbers)
                print(f"Success: {file_path} - {str(line_numbers)}")
            else:
                print(f"Something went wrong: empty line numbers set for {file_path}")
        else:
            print(f"Something went wrong: empty source code for {file_path}")

In [26]:
extract_and_update_vulnerable_lines_in_df(df)

Success: D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\smartbugs-curated-v2\access_control\arbitrary_location_write_simple.sol - {27}
Success: D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\smartbugs-curated-v2\access_control\FibonacciBalance.sol - {38, 31}
Success: D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\smartbugs-curated-v2\access_control\incorrect_constructor_name1.sol - {20}
Success: D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\smartbugs-curated-v2\access_control\incorrect_constructor_name2.sol - {18}
Success: D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\smartbugs-curated-v2\access_control\incorrect_constructor_name3.sol - {17}
Success: D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\smartbugs-curated-v2\access_control\mapping_write.sol - {20}
Success: D:\UoM\Final_Year_Projec

In [28]:
df

,subfolder,filename,source_code,vuln_lines
0,access_control,arbitrary_location_write_simple.sol,/*\n * @source: https://smartcontractsecurity....,{27}
1,access_control,FibonacciBalance.sol,/*\n * @source: https://github.com/sigp/solidi...,"{38, 31}"
2,access_control,incorrect_constructor_name1.sol,/*\n * @source: https://github.com/trailofbits...,{20}
3,access_control,incorrect_constructor_name2.sol,/*\n * @source: https://smartcontractsecurity....,{18}
4,access_control,incorrect_constructor_name3.sol,/*\n * @source: https://smartcontractsecurity....,{17}
...,...,...,...,...
142,unchecked_low_level_calls,etherpot_lotto.sol,/*\n * @source: https://github.com/etherpot/co...,"{109, 141}"
143,unchecked_low_level_calls,king_of_the_ether_throne.sol,/*\n * @source: https://github.com/kieranelby/...,"{118, 132, 174, 110}"
144,unchecked_low_level_calls,lotto.sol,/*\n * @source: https://github.com/sigp/solidi...,"{27, 20}"
145,unchecked_low_level_calls,mishandled.sol,/*\n * @source: https://github.com/seresistvan...,{14}


In [30]:
(df['source_code'] == None).sum()

0

In [32]:
(df['vuln_lines'] == None).sum()

0

In [34]:
!mkdir processed

A subdirectory or file processed already exists.


In [36]:
df.to_csv("processed/smartbugs_curated_dataset_with_vuln_line_numbers_v2.csv")